# TED 03 — Notebook 02: Treinamento, seleção e avaliação dos modelos
**Pergunta-guia:** um modelo de ML consegue prever a categoria de falha a partir do sintoma relatado melhor do que as probabilidades a priori da TED 02?

Fluxo: linhas de base → 6 modelos candidatos (grade de hiperparâmetros por validação cruzada no treino) → escolha do modelo → **uma única avaliação final** no teste, com intervalos de confiança.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import modelagem as M, graficos as g
g.aplicar_estilo()
ROOT = Path("../..").resolve()
RJ = ROOT / "reports" / "resultados_modelagem.json"
R = json.loads(RJ.read_text()) if RJ.exists() else {}
def salvar_R(): RJ.write_text(json.dumps(R, ensure_ascii=False, indent=1, default=float))
import time
import joblib
from sklearn.base import clone
from sklearn.metrics import classification_report, confusion_matrix, log_loss
from matplotlib.colors import LinearSegmentedColormap
df = M.carregar_dados(); tr, te = M.dividir_temporal(df)
COLS = [M.COL_TEXTO, M.COL_APARELHO]
XTR, XTE, ytr, yte = tr[COLS], te[COLS], tr[M.ALVO], te[M.ALVO]
CLASSES = np.array(sorted(df[M.ALVO].unique()))
print(len(tr), len(te), len(CLASSES))

2224 1795 13


## 1. Linhas de base
Sem uma referência, uma acurácia de 70% não significa nada. Usamos duas: (a) **classe majoritária** e (b) **prior por aparelho** — a ideia proposta na TED 02 de usar a distribuição de falhas por tipo de aparelho como probabilidade inicial. Qualquer modelo que use o texto precisa superar a (b) para se justificar.

In [2]:
linhas, preds, scrs = [], {}, {}
for nome, mdl in [("Base · classe majoritária", M.PriorPorAparelho(False)), ("Base · prior por aparelho", M.PriorPorAparelho(True))]:
    mdl.fit(XTR, ytr); p = mdl.predict(XTE); s = mdl.predict_proba(XTE)
    preds[nome], scrs[nome] = p, s
    linhas.append({"modelo": nome, "cv_f1_macro": np.nan, "cv_dp": np.nan, "hiperparametros": "—", **M.metricas(yte, p, s, mdl.classes_)})
pd.DataFrame(linhas)[["modelo", "acuracia", "f1_macro", "top3"]].round(3)

,modelo,acuracia,f1_macro,top3
0,Base · classe majoritária,0.145,0.020,0.526
1,Base · prior por aparelho,0.175,0.053,0.534


## 2. Modelos candidatos e validação cruzada
Todos são modelos **lineares sobre TF-IDF**. A escolha é intencional: com ~2.200 textos curtos, modelos lineares são os mais adequados (poucos dados, muitas variáveis esparsas), rápidos de treinar e interpretáveis; redes neurais/Transformers exigiriam bem mais dados e infraestrutura. Candidatos:
1. **Regressão logística (LR)** com palavras · 2. LR com caracteres · 3. LR com palavras+caracteres · 4. LR com palavras+caracteres **+ aparelho** · 5. **SVM linear** · 6. **Naive Bayes (Complement)**, indicado para texto com classes desbalanceadas.
Métrica de seleção: **F1-macro** (média do F1 de cada classe), porque a acurácia esconderia o desempenho nas classes raras.

In [3]:
cv = M.validacao_cruzada(tr)
ajustados = {}
for nome, (pipe, grade) in M.catalogo_modelos().items():
    t = time.time(); gs = M.treinar_e_selecionar(nome, pipe, grade, tr, cv); ajustados[nome] = gs
    mdl = gs.best_estimator_; p = mdl.predict(XTE); s = M.pontuacoes(mdl, XTE)
    i = gs.best_index_
    preds[nome], scrs[nome] = p, s
    linhas.append({"modelo": nome, "cv_f1_macro": gs.cv_results_["mean_test_score"][i], "cv_dp": gs.cv_results_["std_test_score"][i],
                   "hiperparametros": json.dumps(gs.best_params_), **M.metricas(yte, p, s, mdl.classes_)})
    print(f"{nome:50s} CV F1-macro = {gs.best_score_:.3f} ± {gs.cv_results_['std_test_score'][i]:.3f}  {gs.best_params_}  ({time.time()-t:.0f}s)")
tab = pd.DataFrame(linhas)

LR · palavras                                      CV F1-macro = 0.540 ± 0.028  {'clf__C': 10}  (6s)


LR · caracteres                                    CV F1-macro = 0.597 ± 0.014  {'clf__C': 3}  (22s)


LR · palavras+caracteres                           CV F1-macro = 0.583 ± 0.024  {'clf__C': 1}  (23s)


LR · palavras+caracteres+aparelho                  CV F1-macro = 0.595 ± 0.013  {'clf__C': 3}  (31s)


SVM linear · palavras+caracteres                   CV F1-macro = 0.600 ± 0.028  {'clf__C': 0.3}  (5s)


Naive Bayes (Complement) · palavras+caracteres     CV F1-macro = 0.568 ± 0.025  {'clf__alpha': 0.1}  (4s)


## 3. Escolha do modelo
Critérios derivados dos requisitos do MVP e da EDA e aplicados sobre os resultados da **validação cruzada** (o conjunto de teste não entra na escolha e só é usado para medir o modelo, na seção 4):
1. considerar *empatados* os modelos cujo F1-macro de validação cruzada está a até **1 desvio-padrão** do melhor;
2. entre eles, exigir **saída em probabilidades** — o MVP precisa ordenar causas e informar o grau de confiança (SVM linear só produz margens, não probabilidades);
3. preferir o que usa **todas as informações disponíveis no momento do diagnóstico** (sintoma + tipo de aparelho), já que a TED 02 mostrou associação entre aparelho e falha (V de Cramér = 0,28) e o fluxo do MVP começa pela identificação do equipamento.

In [4]:
cand = tab[tab["cv_f1_macro"].notna()].copy()
melhor = cand.loc[cand["cv_f1_macro"].idxmax()]
elegiveis = cand[cand["cv_f1_macro"] >= melhor["cv_f1_macro"] - melhor["cv_dp"]]
print("Melhor em validação cruzada:", melhor["modelo"], round(melhor["cv_f1_macro"], 3), "± dp", round(melhor["cv_dp"], 3))
print("Empatados (dentro de 1 dp):", list(elegiveis["modelo"]))
FINAL = "LR · palavras+caracteres+aparelho"
assert FINAL in list(elegiveis["modelo"])
R["modelo_final"] = FINAL; R["melhor_cv"] = melhor["modelo"]; R["criterio_empate_dp"] = float(melhor["cv_dp"])
R["grade"] = {k: v[1] for k, v in M.catalogo_modelos().items()}
R["tabela_modelos"] = tab.replace({np.nan: None}).to_dict("records")
print("Modelo escolhido:", FINAL, ajustados[FINAL].best_params_)

Melhor em validação cruzada: SVM linear · palavras+caracteres 0.6 ± dp 0.028
Empatados (dentro de 1 dp): ['LR · caracteres', 'LR · palavras+caracteres', 'LR · palavras+caracteres+aparelho', 'SVM linear · palavras+caracteres']
Modelo escolhido: LR · palavras+caracteres+aparelho {'clf__C': 3}


## 4. Avaliação final no conjunto de teste (eventos de 2019–2020)

In [5]:
mdl = ajustados[FINAL].best_estimator_
p, s = preds[FINAL], scrs[FINAL]; conf = s.max(axis=1)
mf = M.metricas(yte, p, s, mdl.classes_)
mf["log_loss"] = float(log_loss(yte, s, labels=mdl.classes_))
base = "Base · prior por aparelho"
ic = M.bootstrap_ic(yte, p, s, mdl.classes_, base_pred=preds[base])
R["final"] = {"metricas": mf, "ic95": ic, "hiperparametros": ajustados[FINAL].best_params_}
for k in ("acuracia", "f1_macro", "top3"): print(f"{k:10s} = {mf[k]:.3f}   IC95% {ic[k][0]:.3f}–{ic[k][1]:.3f}")
print("Ganho de F1-macro sobre o prior por aparelho (IC95%):", [round(x, 3) for x in ic["dif_f1_macro_vs_base"]])
print("log-loss:", round(mf["log_loss"], 3), "(referência do prior por aparelho:", round(log_loss(yte, scrs[base], labels=CLASSES), 3), ")")
R["final"]["log_loss_base"] = float(log_loss(yte, scrs[base], labels=CLASSES))
tab[["modelo", "cv_f1_macro", "cv_dp", "acuracia", "f1_macro", "top3"]].round(3)

acuracia   = 0.716   IC95% 0.695–0.737
f1_macro   = 0.654   IC95% 0.619–0.683
top3       = 0.895   IC95% 0.881–0.909
Ganho de F1-macro sobre o prior por aparelho (IC95%): [0.567, 0.631]
log-loss: 1.025 (referência do prior por aparelho: 2.236 )


,modelo,cv_f1_macro,cv_dp,acuracia,f1_macro,top3
0,Base · classe majoritária,NaN,NaN,0.145,0.020,0.526
1,Base · prior por aparelho,NaN,NaN,0.175,0.053,0.534
2,LR · palavras,0.540,0.028,0.656,0.566,0.856
3,LR · caracteres,0.597,0.014,0.708,0.636,0.900
4,LR · palavras+caracteres,0.583,0.024,0.696,0.613,0.895
5,LR · palavras+caracteres+aparelho,0.595,0.013,0.716,0.654,0.895
6,SVM linear · palavras+caracteres,0.600,0.028,0.710,0.616,0.892
7,Naive Bayes (Complement) · palavras+caracteres,0.568,0.025,0.699,0.584,0.883


In [6]:
# Comparação visual de todos os modelos (fig14)
t2 = tab.copy(); t2 = t2.sort_values("f1_macro")
fig, ax = plt.subplots(figsize=(9, 4.8)); y = np.arange(len(t2))
ax.barh(y + .2, t2["f1_macro"], .38, color=g.SERIES[1], label="Teste (2019–2020)")
ax.barh(y - .2, t2["cv_f1_macro"].fillna(0), .38, xerr=t2["cv_dp"].fillna(0), color=g.SERIES[0], label="Validação cruzada (treino)", error_kw={"ecolor": g.TXT2, "lw": 1})
for i, (v, c) in enumerate(zip(t2["f1_macro"], t2["cv_f1_macro"])):
    ax.text(v + .008, i + .2, f"{v:.2f}", va="center", fontsize=8.5, color=g.TXT2)
    if not np.isnan(c): ax.text(c + .02, i - .2, f"{c:.2f}", va="center", fontsize=8.5, color=g.TXT2)
ax.set_yticks(y); ax.set_yticklabels([("★ " if m == FINAL else "") + m for m in t2["modelo"]], fontsize=8.5); ax.grid(axis="y", visible=False)
ax.set_xlim(0, .8); ax.set_xlabel("F1-macro"); ax.set_title("Comparação dos modelos (★ = modelo escolhido)"); ax.legend(loc="lower right")
fig.tight_layout(); g.salvar(fig, "fig14_comparacao_modelos"); plt.show()

In [7]:
# Desempenho por classe (fig15) e relatório
rel = pd.DataFrame(classification_report(yte, p, output_dict=True, zero_division=0)).T.loc[list(CLASSES)]
relb = pd.DataFrame(classification_report(yte, preds[base], output_dict=True, zero_division=0)).T.loc[list(CLASSES)]
R["por_classe"] = {c: {"precisao": float(rel.loc[c, "precision"]), "revocacao": float(rel.loc[c, "recall"]), "f1": float(rel.loc[c, "f1-score"]),
                       "n": int(rel.loc[c, "support"]), "f1_base": float(relb.loc[c, "f1-score"]), "n_treino": int((ytr == c).sum())} for c in CLASSES}
o = rel.sort_values("f1-score"); fig, ax = plt.subplots(figsize=(8.6, 5.2)); y = np.arange(len(o))
ax.barh(y + .2, o["f1-score"], .38, color=g.SERIES[0], label="Modelo escolhido"); ax.barh(y - .2, relb.loc[o.index, "f1-score"], .38, color=g.CINZA, label="Prior por aparelho")
for i, c in enumerate(o.index): ax.text(max(o.loc[c, "f1-score"], .02) + .01, i + .2, f"{o.loc[c, 'f1-score']:.2f}  (n={int(o.loc[c, 'support'])})", va="center", fontsize=8, color=g.TXT2)
ax.set_yticks(y); ax.set_yticklabels(o.index, fontsize=8.5); ax.grid(axis="y", visible=False); ax.set_xlim(0, 1.0); ax.set_xlabel("F1 no teste"); ax.set_title("F1 por categoria de falha"); ax.legend(loc="lower right")
fig.tight_layout(); g.salvar(fig, "fig15_f1_por_classe"); plt.show()
rel.round(3)

,precision,recall,f1-score,support
Armazenamento,0.619,0.703,0.658,74.0
Botões / teclado / touchpad,0.750,0.758,0.754,95.0
Carcaça / dano externo,0.562,0.574,0.568,47.0
Conectores e portas,0.750,0.785,0.767,130.0
Dano por líquido,0.714,0.455,0.556,11.0
Desempenho / superaquecimento,0.611,0.806,0.695,160.0
Energia / bateria,0.819,0.738,0.776,275.0
Inicialização (boot),0.552,0.630,0.589,92.0
Outros,0.303,0.230,0.261,87.0
Placa / dano interno,0.441,0.506,0.471,89.0


In [8]:
# Matriz de confusão normalizada pela classe verdadeira (fig16)
cm = confusion_matrix(yte, p, labels=CLASSES); cmn = cm / cm.sum(axis=1, keepdims=True)
cmap = LinearSegmentedColormap.from_list("az", g.AZUL_SEQ)
fig, ax = plt.subplots(figsize=(9.2, 7.6)); im = ax.imshow(cmn, cmap=cmap, vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if cmn[i, j] >= .05: ax.text(j, i, f"{cmn[i,j]*100:.0f}", ha="center", va="center", fontsize=7.5, color="white" if cmn[i, j] > .5 else g.TXT)
ax.set_xlabel("Categoria prevista"); ax.set_ylabel("Categoria verdadeira"); ax.set_title("Matriz de confusão (% de cada classe verdadeira)")
fig.colorbar(im, ax=ax, fraction=.04, pad=.02); fig.tight_layout(); g.salvar(fig, "fig16_matriz_confusao"); plt.show()
pares = [(CLASSES[i], CLASSES[j], int(cm[i, j]), float(cmn[i, j])) for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j]
pares = sorted(pares, key=lambda x: -x[2])[:8]
R["confusoes"] = [{"verdadeiro": a, "previsto": b, "n": n, "pct_da_classe": pc} for a, b, n, pc in pares]
pd.DataFrame(R["confusoes"])

,verdadeiro,previsto,n,pct_da_classe
0,Software / sistema,Desempenho / superaquecimento,43,0.105392
1,Energia / bateria,Inicialização (boot),22,0.080000
2,Software / sistema,Outros,21,0.051471
3,Outros,Software / sistema,18,0.206897
4,Energia / bateria,Conectores e portas,14,0.050909
5,Outros,Desempenho / superaquecimento,12,0.137931
6,Placa / dano interno,Energia / bateria,11,0.123596
7,Desempenho / superaquecimento,Software / sistema,10,0.062500


In [9]:
# Top-k: a causa certa está entre as k mais prováveis? (fig17) — é o que importa para ordenar os testes do MVP
ks = [1, 2, 3, 4, 5]
curvas = {n: [M.topk_acuracia(yte.values, scrs[n], CLASSES, k) for k in ks] for n in [FINAL, base, "Base · classe majoritária"]}
R["topk"] = {n: dict(zip(map(str, ks), v)) for n, v in curvas.items()}
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for (n, v), c, ls in zip(curvas.items(), [g.SERIES[0], g.SERIES[1], g.CINZA], ["-", "-", "--"]):
    ax.plot(ks, np.array(v) * 100, marker="o", color=c, ls=ls, label=n.replace("Base · ", "Base: "))
    ax.text(5.08, v[-1] * 100, f"{v[-1]*100:.0f}%", va="center", fontsize=8.5, color=c)
ax.set_xticks(ks); ax.set_xlabel("k (nº de causas sugeridas)"); ax.set_ylabel("% de casos com a causa correta entre as k primeiras"); ax.set_ylim(0, 105); ax.set_xlim(.8, 5.5)
ax.set_title("Acerto top-k no teste"); ax.legend(loc="lower right", fontsize=8.5); fig.tight_layout(); g.salvar(fig, "fig17_topk"); plt.show()
pd.DataFrame(curvas, index=[f"top-{k}" for k in ks]).round(3).T

,top-1,top-2,top-3,top-4,top-5
LR · palavras+caracteres+aparelho,0.716,0.841,0.895,0.928,0.949
Base · prior por aparelho,0.175,0.412,0.534,0.642,0.711
Base · classe majoritária,0.145,0.373,0.526,0.615,0.687


## 5. Salvando modelo e predições
- `models/classificador_falhas_treino2018.joblib` — treinado só com o treino (é o que foi avaliado acima).
- `models/classificador_falhas.joblib` — mesmos hiperparâmetros, **reajustado com todos os 4.019 registros**, para uso no protótipo do MVP (prática padrão após a avaliação).

In [10]:
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump(mdl, ROOT / "models" / "classificador_falhas_treino2018.joblib", compress=3)
final_all = clone(mdl).fit(df[COLS], df[M.ALVO]); joblib.dump(final_all, ROOT / "models" / "classificador_falhas.joblib", compress=3)
import os
R["tamanho_modelo_mb"] = round(os.path.getsize(ROOT / "models" / "classificador_falhas.joblib") / 1e6, 2)
pred_df = te[["id_origem", "categoria", "ano_evento", "problema", M.ALVO]].copy(); pred_df["prevista"] = p; pred_df["confianca"] = conf.round(4)
pred_df["top3"] = [" | ".join(mdl.classes_[np.argsort(-r)[:3]]) for r in s]
pred_df.to_csv(ROOT / "data" / "processed" / "predicoes_teste_ted03.csv", index=False)
salvar_R(); print("modelo (MB):", R["tamanho_modelo_mb"])

modelo (MB): 2.27
